In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

from utils import model_untils


In [7]:
from langgraph.checkpoint.memory import InMemorySaver


def dynamic_model_selection_demo():
    """
    根据对话复杂性动态选择模型

    核心思路：
      - 简单问题 / 短对话 → 使用轻量模型（本地 Ollama），速度快、成本低
      - 复杂问题 / 长对话 → 使用高级模型（云端 Qwen），能力更强

    通过 middleware（中间件）机制，在每次模型调用前自动判断并切换模型
    """
    print("=" * 60)
    print("示例 3: 动态选择模型 - 根据对话复杂度自动切换")
    print("=" * 60)

    from langchain_core.tools import tool

    # 定义工具
    @tool
    def get_weather(location: str) -> str:
        """获取指定位置的天气信息。"""
        weather_db = {
            "北京": "晴，25°C",
            "上海": "多云，28°C",
            "广州": "小雨，30°C",
            "深圳": "晴，29°C",
        }
        return weather_db.get(location, f"{location} 的天气：暂无数据")

    @tool
    def calculator(expression: str) -> str:
        """计算数学表达式的结果。"""
        try:
            return f"计算结果：{eval(expression)}"
        except Exception as e:
            return f"计算错误：{e}"

    # -------------------------------------------------------------------------
    # 准备两个模型：基础模型 + 高级模型
    # -------------------------------------------------------------------------
    basic_model = ChatOllama(model="qwen3.5:2b")           # 本地轻量模型
    advanced_model = model_untils.get_qwen_client()        # 云端高级模型

    if advanced_model is None:
        print("【跳过】未配置阿里云 API Key，无法运行此示例")
        return

    # -------------------------------------------------------------------------
    # 定义中间件：根据消息数量动态选择模型
    # -------------------------------------------------------------------------
    @wrap_model_call
    def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
        """根据对话复杂性选择模型。

        策略：
          - 对话消息数 > 10 → 切换到高级模型（长对话需要更强的推理能力）
          - 对话消息数 ≤ 10 → 使用基础模型（简单任务不需要大模型）
        """
        message_count = len(request.state["messages"])

        if message_count > 10:
            # 对较长的对话使用高级模型
            model = advanced_model
            print(f"  [动态切换] 消息数={message_count} → 使用高级模型 (Qwen)")
        else:
            model = basic_model
            print(f"  [动态切换] 消息数={message_count} → 使用基础模型 (Ollama)")

        request.model = model
        return handler(request)

    # -------------------------------------------------------------------------
    # 创建带中间件的 Agent
    # -------------------------------------------------------------------------
    agent = create_agent(
        model=basic_model,                        # 默认模型
        tools=[get_weather, calculator],
        middleware=[dynamic_model_selection],       # 注入动态选择中间件
        checkpointer=InMemorySaver()
    )

    print("【Agent 创建成功 - 动态模型选择模式】")
    print(f"  基础模型: Ollama (qwen3.5:2b) - 本地轻量")
    print(f"  高级模型: Qwen (qwen-plus) - 云端强大")
    print(f"  切换策略: 消息数 > 10 时切换到高级模型")
    print(f"  工具 1: get_weather")
    print(f"  工具 2: calculator")
    print()

    # 调用 Agent
    questions = [
        "我是Luke，很高兴认识你"
        "北京今天天气怎么样？",
        "23 加 45 等于多少？",
        "我叫什么名字?"
    ]

    for question in questions:
        print(f"【用户提问】{question}")
        try:
            result = agent.invoke(
                {"messages": [{"role": "user", "content": question}]},
                {"configurable": {"thread_id": "1"}}
            )
            last_msg = result["messages"][-1]
            return last_msg.content
            print(f"【Agent 回答】{last_msg.content}")
        except Exception as e:
            print(f"【执行出错】{e}")
        print()

In [8]:
dynamic_model_selection_demo()

示例 3: 动态选择模型 - 根据对话复杂度自动切换
【Agent 创建成功 - 动态模型选择模式】
  基础模型: Ollama (qwen3.5:2b) - 本地轻量
  高级模型: Qwen (qwen-plus) - 云端强大
  切换策略: 消息数 > 10 时切换到高级模型
  工具 1: get_weather
  工具 2: calculator

【用户提问】我是Luke，很高兴认识你北京今天天气怎么样？
  [动态切换] 消息数=1 → 使用基础模型 (Ollama)


C:\Users\jwjxl\AppData\Local\Temp\ipykernel_18968\111528665.py:71: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = model


  [动态切换] 消息数=3 → 使用基础模型 (Ollama)


C:\Users\jwjxl\AppData\Local\Temp\ipykernel_18968\111528665.py:71: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = model


'北京今天的天气非常好！阳光明媚，气温25°C，非常适合户外活动和游玩。\n\n不过，您刚才提到了您的名字叫Luke，看来我可能有点弄混了。不过没关系，我已经记住了！如果您还有其他问题，随时都可以问我。'

In [4]:
dynamic_model_selection_demo("我是Luke，很高兴认识你")

示例 3: 动态选择模型 - 根据对话复杂度自动切换
【Agent 创建成功 - 动态模型选择模式】
  基础模型: Ollama (qwen3.5:2b) - 本地轻量
  高级模型: Qwen (qwen-plus) - 云端强大
  切换策略: 消息数 > 10 时切换到高级模型
  工具 1: get_weather
  工具 2: calculator

【用户提问】我是Luke，很高兴认识你
  [动态切换] 消息数=1 → 使用基础模型 (Ollama)


C:\Users\jwjxl\AppData\Local\Temp\ipykernel_36364\608073452.py:68: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = model


'你好，Luke！很高兴认识你。有什么我可以帮你的吗？'

In [5]:
dynamic_model_selection_demo("深圳今天天气怎么样？")

示例 3: 动态选择模型 - 根据对话复杂度自动切换
【Agent 创建成功 - 动态模型选择模式】
  基础模型: Ollama (qwen3.5:2b) - 本地轻量
  高级模型: Qwen (qwen-plus) - 云端强大
  切换策略: 消息数 > 10 时切换到高级模型
  工具 1: get_weather
  工具 2: calculator

【用户提问】深圳今天天气怎么样？
  [动态切换] 消息数=1 → 使用基础模型 (Ollama)


C:\Users\jwjxl\AppData\Local\Temp\ipykernel_36364\608073452.py:68: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = model


  [动态切换] 消息数=3 → 使用基础模型 (Ollama)


C:\Users\jwjxl\AppData\Local\Temp\ipykernel_36364\608073452.py:68: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = model


'深圳今天天气晴，气温为29°C。'

In [6]:
dynamic_model_selection_demo("上海呢?")

示例 3: 动态选择模型 - 根据对话复杂度自动切换
【Agent 创建成功 - 动态模型选择模式】
  基础模型: Ollama (qwen3.5:2b) - 本地轻量
  高级模型: Qwen (qwen-plus) - 云端强大
  切换策略: 消息数 > 10 时切换到高级模型
  工具 1: get_weather
  工具 2: calculator

【用户提问】上海呢?
  [动态切换] 消息数=1 → 使用基础模型 (Ollama)


C:\Users\jwjxl\AppData\Local\Temp\ipykernel_36364\608073452.py:68: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = model


  [动态切换] 消息数=3 → 使用基础模型 (Ollama)


C:\Users\jwjxl\AppData\Local\Temp\ipykernel_36364\608073452.py:68: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = model


'上海目前天气状况是多云，气温 28°C。'

In [9]:
dynamic_model_selection_demo("机器学习是什么概念？")

示例 3: 动态选择模型 - 根据对话复杂度自动切换
【Agent 创建成功 - 动态模型选择模式】
  基础模型: Ollama (qwen3.5:2b) - 本地轻量
  高级模型: Qwen (qwen-plus) - 云端强大
  切换策略: 消息数 > 10 时切换到高级模型
  工具 1: get_weather
  工具 2: calculator

【用户提问】机器学习是什么概念？
  [动态切换] 消息数=1 → 使用基础模型 (Ollama)


C:\Users\jwjxl\AppData\Local\Temp\ipykernel_36364\608073452.py:68: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = model


'机器学习（Machine Learning，简称ML）是计算机科学领域中一个重要的子领域，主要研究如何使系统能够从数据中学习、建模和预测。\n\n## 核心概念\n\n### 1. 基本原理\n机器学习是人工智能的重要组成部分，它使计算机系统能够通过观察和积累数据来"学习"，而无需被明确地编程。\n\n### 2. 机器学习的关键组成部分\n- **数据**：机器学习需要大量的高质量数据作为基础\n- **算法**：特定的数学模型和计算规则\n- **训练**：通过训练让系统从数据中学习规律\n- **模型**：经过训练的系统，可以处理新数据\n\n### 3. 学习过程\n机器学习包含以下几个阶段：\n1. **输入数据**：原始数据集\n2. **特征提取**：从数据中提取出关键信息\n3. **模型训练**：系统学习数据规律\n4. **模型评估**：验证模型效果\n5. **部署应用**：将模型用于实际场景\n\n### 4. 机器学习与传统编程的区别\n\n| 传统编程 | 机器学习 |\n|---------|---------|\n| 需要预先定义规则 | 从数据中学习规律 |\n| 基于规则 | 需要大量数据 |\n| 结果确定 | 结果可能不确定 |\n| 编程者定义逻辑 | 自动调整参数 |\n\n### 5. 常见应用场景\n- **推荐系统**：如短视频推荐、电商商品推荐\n- **语音识别**：将语音转换为文字\n- **图像识别**：识别物体、分类图片\n- **自然语言处理**：聊天机器人、翻译\n- **医疗诊断**：疾病预测和分析\n- **自动驾驶**：车辆识别和环境感知\n\n### 6. 为什么机器学习很重要？\n- 能够处理复杂的数据模式\n- 需要大量数据才能训练成功\n- 能够适应新数据和变化\n- 自动优化，提高效率\n\n机器学习正从学术研究和实验室开发，走向实际应用，正在改变各行各业的生产力。'